In [ ]:
using Plots, Polynomials, LaTeXStrings, ColorSchemes, DelimitedFiles, DataFrames, Interpolations
using Statistics, StatsPlots, Random, ProgressMeter, Printf, LinearAlgebra, Plots.PlotMeasures
include("utils.jl")

In [ ]:
# Modifying backend GR attributes
gr(guidefontsize=25, tickfontsize=20, legendfontsize=25, margin=5Plots.mm, grid=false)
myApple = RGBA(187/255, 206/255, 131/255, 1)
mySalmon = RGBA(243/255, 124/255, 130/255)
myYellow = RGBA(228/255, 205/255, 121/255, 1)
myBlue = RGBA(131/255, 174/255, 218/255, 1)
myDarkBlue = RGBA(114/255, 119/255, 217/255, 1)
myOrange = RGBA(241/255, 175/255, 113/255, 1)
myPink = RGBA(243/255, 124/255, 130/255, 1)
myPurple = RGBA(169/255, 90/255, 179/255, 1)
myGreen = RGBA(132/255, 195/255, 168/255, 1)
myRed = RGBA(158/255, 3/255, 8/255, 1)
myGray = RGBA(150/255, 150/255, 150/255, 1)
myLightBlue = RGBA(127/255, 154/255, 209/255, 1);
default(fmt = :png);

# Mapping positive weight to width

In [ ]:
# Get all curves from your 200-curve file
I_input = 0.1:0.1:3
width = range(0.5, 15, length=201)
all_curves_positive = get_all_xy_pairs("./Cadence_output/current_mirror_pos.vcsv")

mapping_positive = zeros(length(I_input), length(width))

for i = 1 : length(I_input)
    outputs = all_curves_positive[i][2]*1e9
    mapping_positive[i, :] = outputs
end

display(mapping_positive)

In [ ]:
heatmap(mapping_positive)

In [ ]:
I_input = 0.1:0.1:3.0  # nA
width = range(0.5, 15, length=201)  # μm

# Build interpolator: (I_in, width) → I_out
itp_positive = interpolate(mapping_positive, BSpline(Cubic(Line(OnGrid()))))
sitp_positive = scale(itp_positive, I_input, width)

function find_width_positive(I_in::Float64, weight::Float64; 
                    width_range=width, tol=1e-6, max_iter=100)
    """
    Given input current I_in (nA) and desired weight,
    find the transistor width (μm) that produces I_out = weight × I_in
    """
    I_out_desired = weight * I_in
    
    w_min, w_max = first(width_range), last(width_range)
    
    I_out_at_w_min = sitp_positive(I_in, w_min)
    I_out_at_w_max = sitp_positive(I_in, w_max)
    
    # Global slope across entire data range
    slope = (I_out_at_w_max - I_out_at_w_min) / (w_max - w_min)
    
    # Extrapolate linearly if outside range
    if I_out_desired < I_out_at_w_min
        # Linear extrapolation below
        w_result = w_min + (I_out_desired - I_out_at_w_min) / slope
        return w_result  # clamp to min physical width
    elseif I_out_desired > I_out_at_w_max
        # Linear extrapolation above
        w_result = w_max + (I_out_desired - I_out_at_w_max) / slope
        return w_result
    end
    
    # Within range: binary search with interpolator
    w_lo, w_hi = w_min, w_max
    
    for _ in 1:max_iter
        w_mid = (w_lo + w_hi) / 2
        I_out_mid = sitp_positive(I_in, w_mid)
        
        if abs(I_out_mid - I_out_desired) < tol
            return w_mid
        elseif I_out_mid < I_out_desired
            w_lo = w_mid
        else
            w_hi = w_mid
        end
    end
    
    return (w_lo + w_hi) / 2
end

In [ ]:
# Example usage
I_in = 0.559198379516602     # nA
weight = 0.2686214149 # from your W_cand matrix

w = find_width_positive(I_in, weight)
println("For I_in = $I_in nA and weight = $weight")
println("Required width = $(round(w, digits=3)) μm")

# Mapping negative weight to width

In [ ]:
# Get all curves from your 200-curve file
I_input = 0.1:0.1:3
width = range(0.5, 15, length=201)
all_curves_negative = get_all_xy_pairs("./Cadence_output/current_mirror_neg.vcsv")

mapping_negative = zeros(length(I_input), length(width))

for i = 1 : length(I_input)
    outputs = all_curves_negative[i][2]*1e9
    mapping_negative[i, :] = outputs
end

display(mapping_negative)

In [ ]:
heatmap(mapping_negative)

In [ ]:
I_input = 0.1:0.1:3.0  # nA
width = range(0.5, 15, length=201)  # μm

# Build interpolator: (I_in, width) → I_out
itp_negative = interpolate(mapping_negative, BSpline(Cubic(Line(OnGrid()))))
sitp_negative = scale(itp_negative, I_input, width)

function find_width_negative(I_in::Float64, weight::Float64; 
                    width_range=width, tol=1e-6, max_iter=100)
    """
    Given input current I_in (nA) and desired weight,
    find the transistor width (μm) that produces I_out = weight × I_in
    """
    I_out_desired = weight * I_in
    
    w_min, w_max = first(width_range), last(width_range)
    
    I_out_at_w_min = sitp_negative(I_in, w_min)
    I_out_at_w_max = sitp_negative(I_in, w_max)
    
    # Global slope across entire data range
    slope = (I_out_at_w_max - I_out_at_w_min) / (w_max - w_min)
    
    # Extrapolate linearly if outside range
    if I_out_desired < I_out_at_w_min
        # Linear extrapolation below
        w_result = w_min + (I_out_desired - I_out_at_w_min) / slope
        return w_result  # clamp to min physical width
    elseif I_out_desired > I_out_at_w_max
        # Linear extrapolation above
        w_result = w_max + (I_out_desired - I_out_at_w_max) / slope
        return w_result
    end
    
    # Within range: binary search with interpolator
    w_lo, w_hi = w_min, w_max
    
    for _ in 1:max_iter
        w_mid = (w_lo + w_hi) / 2
        I_out_mid = sitp_negative(I_in, w_mid)
        
        if abs(I_out_mid - I_out_desired) < tol
            return w_mid
        elseif I_out_mid < I_out_desired
            w_lo = w_mid
        else
            w_hi = w_mid
        end
    end
    
    return (w_lo + w_hi) / 2
end

In [ ]:
# Example usage
I_in = 0.559198379516602      # nA
weight = 0.0651067644   # from your W_cand matrix

w = find_width_negative(I_in, weight)
println("For I_in = $I_in nA and weight = $weight")
println("Required width = $(round(w, digits=3)) μm")